## Start Spark + Delta session, read Bronze tables back

Here the input is Bronze (already-ingested data), not the raw CSVs directly

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
 # setup directories
PROJECT_ROOT = Path.cwd().parent
BRONZE_DIR = (PROJECT_ROOT / "bronze").as_posix()
SILVER_DIR = (PROJECT_ROOT / "silver").as_posix()

builder = (
    SparkSession.builder
    .appName("movielens-part2")
    .master("local[2]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

Spark version: 3.5.8


In [2]:
# read bronze data
bronze_ratings = spark.read.format("delta").load(f"{BRONZE_DIR}/bronze_ratings")
bronze_movies  = spark.read.format("delta").load(f"{BRONZE_DIR}/bronze_movies")

print("bronze_ratings:", bronze_ratings.count(), "rows")
bronze_ratings.printSchema()

bronze_ratings: 403344 rows
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [3]:
# print bronze_movies count and schema
bronze_movies.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



## Type the column

- **Problem**: `timestamp` is a plain number instead of being time stamp, and `genre` as a string instead of a list.

To fix `timestamp`:
- Use `F.timestamp_seconds("timestamp")` to convert it to the accurate data type
- Rename it to `rated_at` using `.withColumn("rated_at", ...)` for clarity
- Remove the old raw integer version with `.drop("timestamp")`

To fix `genre`:
- `F.split("genres", "\\|")` — splits the genres string on the `|` character, turning `"Comedy|Romance"` into an actual list: `["Comedy", "Romance"]`. The `\\|` (rather than plain `|`) is needed because `|` is a special character in the pattern language split uses (it normally means "or") — escaping it tells Spark "I mean a literal pipe character, not 'or'."


In [4]:
from pyspark.sql import functions as F

ratings_typed = (
    bronze_ratings
    .withColumn("rated_at", F.timestamp_seconds("timestamp"))
    .drop("timestamp")
)

movies_typed = (
    bronze_movies
    .withColumn("genres", F.split("genres", "\\|"))
)

ratings_typed.printSchema()
movies_typed.printSchema()

ratings_typed.select("userId", "movieId", "rating", "rated_at").show(5, truncate=False)
movies_typed.select("movieId", "title", "genres").show(5, truncate=False)

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- rated_at: timestamp (nullable = true)

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- batch_id: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

+------+-------+------+-------------------+
|userId|movieId|rating|rated_at           |
+------+-------+------+-------------------+
|1     |1      |4.0   |2000-07-30 21:45:03|
|1     |3      |4.0   |2000-07-30 21:20:47|
|1     |6      |4.0   |2000-07-30 21:37:04|
|1     |47     |5.0   |2000-07-30 22:03:35|
|1     |50     |5.0   |2000-07-30 21:48:51|
+------+-------+------+-------------------+
only showin

## Manufacture a small synthetic duplicate batch
Real `MovieLens` has zero natural duplicates on (`userId`, `movieId`). So there's nothing for a dedup rule to practice on unless we create a realistic test case ourselves — the brief calls this out explicitly as a "disclosed simplification, not a shortcut."

`ratings_typed` currently has 403,344 rows — that's 4 identical copies of every rating, left over from Part 1's replay tests. So we already have one kind of duplicate (exact copies from repeated ingestion) sitting in there. What we're adding now is a second, different kind: a row where the values themselves changed (same user, same movie, but a new rating and a later timestamp) — simulating someone genuinely re-rating a movie. Step 3's dedup rule will need to handle both kinds at once, which is a good real-world test.

In [5]:
# pick 3 (userId, movieId) pairs to simulate as "updated" ratings
# collapses down to one row per distinct pair, temporarily, just so we can cleanly pick 3 different users/movies
distinct_ratings = ratings_typed.dropDuplicates(["userId", "movieId"])  
sample_pairs = distinct_ratings.select("userId", "movieId").limit(3)   

synthetic_updates = (
    distinct_ratings
    .join(sample_pairs, on=["userId", "movieId"], how="inner")
    .withColumn("rating", F.lit(5.0))                      # pretend they changed their mind, gave it a 5
    .withColumn("rated_at", F.col("rated_at") + F.expr("INTERVAL 1 DAY"))  # a day later
    .withColumn("batch_id", F.lit("synthetic_update"))
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source_file", F.lit("synthetic"))
)

ratings_with_dupes = ratings_typed.unionByName(synthetic_updates)

print("before:", ratings_typed.count(), "| after adding synthetic updates:", ratings_with_dupes.count())

# show one pair's full history: original replay copies + the synthetic update
original_matches = ratings_typed.join(sample_pairs, on=["userId", "movieId"], how="inner")
comparison = original_matches.unionByName(synthetic_updates).orderBy("userId", "movieId", "rated_at")
comparison.select("userId", "movieId", "rating", "rated_at", "batch_id", "source_file").show(truncate=False)

before: 403344 | after adding synthetic updates: 403347
+------+-------+------+-------------------+----------------+-----------+
|userId|movieId|rating|rated_at           |batch_id        |source_file|
+------+-------+------+-------------------+----------------+-----------+
|1     |1208   |4.0   |2000-07-30 21:54:10|20260914_082437 |ratings.csv|
|1     |1208   |4.0   |2000-07-30 21:54:10|20260914_075205 |ratings.csv|
|1     |1208   |4.0   |2000-07-30 21:54:10|20260917_071842 |ratings.csv|
|1     |1208   |4.0   |2000-07-30 21:54:10|20260914_084954 |ratings.csv|
|1     |1208   |5.0   |2000-07-31 21:54:10|synthetic_update|synthetic  |
|1     |1348   |4.0   |2000-07-30 21:56:33|20260914_075205 |ratings.csv|
|1     |1348   |4.0   |2000-07-30 21:56:33|20260914_082437 |ratings.csv|
|1     |1348   |4.0   |2000-07-30 21:56:33|20260914_084954 |ratings.csv|
|1     |1348   |4.0   |2000-07-30 21:56:33|20260917_071842 |ratings.csv|
|1     |1348   |5.0   |2000-07-31 21:56:33|synthetic_update|syntheti

## Dedupe on `(userId, movieId)` using a stated precedence rule
The rule we're implementing: "if the same (`userId`, `movieId`) pair appears more than once, keep whichever row has the latest rated_at — that's the user's most recent actual decision." This handles both kinds of duplicates sitting in `ratings_with_dupes` right now: the 4 identical replay-copies (same `rated_at`, doesn't matter which "wins" since they're identical) and our synthetic update (genuinely later rated_at, so it should win over the older original).

In [6]:
from pyspark.sql.window import Window

precedence_window = Window.partitionBy("userId", "movieId").orderBy(F.col("rated_at").desc())

ratings_silver = (
    ratings_with_dupes
    .withColumn("row_num", F.row_number().over(precedence_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

print("before dedup:", ratings_with_dupes.count(), "| after dedup:", ratings_silver.count())

before dedup: 403347 | after dedup: 100836


In [7]:
# verify that the synthetic updates are present in the silver table
ratings_silver.join(sample_pairs, on=["userId", "movieId"], how="inner") \
    .select("userId", "movieId", "rating", "rated_at", "source_file") \
    .show(truncate=False)

+------+-------+------+-------------------+-----------+
|userId|movieId|rating|rated_at           |source_file|
+------+-------+------+-------------------+-----------+
|1     |1208   |5.0   |2000-07-31 21:54:10|synthetic  |
|1     |1348   |5.0   |2000-07-31 21:56:33|synthetic  |
|4     |457    |5.0   |1999-12-14 13:00:59|synthetic  |
+------+-------+------+-------------------+-----------+



## Referential integrity check against movies

Confirms every rating points at a real movie, a basic trust check before calling this table "clean"

Every movieId in `ratings_silver` should point at a movie that actually exists in the movies table. If it didn't, that'd be a broken reference — a rating for a movie that (as far as our data is concerned) doesn't exist.

In [8]:
# perform a left anti join to find ratings that don't have a corresponding movie in the movies table
orphaned_ratings = ratings_silver.join(movies_typed, on="movieId", how="left_anti")

print("orphaned ratings (movieId not in movies table):", orphaned_ratings.count())

orphaned ratings (movieId not in movies table): 0


## Prove reruns of the Silver build don't duplicate.
1. Write `ratings_silver` to disk for the first time (as if this represents "everything we've processed so far")
2. Simulate a few brand-new ratings arriving later (not updates — new people rating new movies), rerun the whole Silver build, and confirm the new rows show up correctly
3. Rerun the exact same build again, with no new data at all, and confirm nothing changes — proving a rerun is safe

In [9]:
# write the silver ratings to disk
ratings_silver.write.format("delta").mode("overwrite").save(f"{SILVER_DIR}/silver_ratings")

check_v1 = spark.read.format("delta").load(f"{SILVER_DIR}/silver_ratings")
print("silver_ratings after first write:", check_v1.count(), "rows")

silver_ratings after first write: 100836 rows


In [10]:
# simulate 2 brand-new ratings arriving later (new user/movie pairs, not updates)
late_batch = (
    spark.range(2)                                            # a tiny built-in Spark table: id = 0, 1
    .withColumn("userId", F.lit(9999).cast("int"))
    .withColumn("movieId", (F.col("id") + 1).cast("int"))     # id=0 -> movieId 1, id=1 -> movieId 2
    .withColumn("rating", F.when(F.col("id") == 0, 4.5).otherwise(3.0))
    .drop("id")
    .withColumn("rated_at", F.current_timestamp())
    .withColumn("batch_id", F.lit("late_batch"))
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source_file", F.lit("late_arrival"))
)

ratings_with_late = ratings_with_dupes.unionByName(late_batch)

ratings_silver_v2 = (
    ratings_with_late
    .withColumn("row_num", F.row_number().over(precedence_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

ratings_silver_v2.write.format("delta").mode("overwrite").save(f"{SILVER_DIR}/silver_ratings")

check_v2 = spark.read.format("delta").load(f"{SILVER_DIR}/silver_ratings")
print("silver_ratings after rerun with late-arriving data:", check_v2.count(), "rows")

silver_ratings after rerun with late-arriving data: 100838 rows


In [11]:
# rerun the exact same build on the exact same input, no new data
ratings_silver_v3 = (
    ratings_with_late
    .withColumn("row_num", F.row_number().over(precedence_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

ratings_silver_v3.write.format("delta").mode("overwrite").save(f"{SILVER_DIR}/silver_ratings")

check_v3 = spark.read.format("delta").load(f"{SILVER_DIR}/silver_ratings")
print("rows after rerun with no new data:", check_v3.count())
print("distinct (userId, movieId) pairs: ", check_v3.dropDuplicates(["userId", "movieId"]).count())

rows after rerun with no new data: 100838
distinct (userId, movieId) pairs:  100838
